# 07 — Run the references on their own

**Optional.** Since `03_references` uses the Semantic Scholar batch endpoint (100 papers per request),
`05_run_pipeline.ipynb` fetches the references itself in a background thread. Use this notebook only
if a teammate should fetch them on their own Colab account (with their own `S2_API_KEY`), for example
to re-fetch references later; then set `RUN_REFERENCES = False` in 05. References need neither the
sources nor the discovery, so this can start at any time.

What it does:
1. Loads `00_config`, `01_discovery` (ID list) and `03_references` (the crawler).
2. IDs: the discovery manifest if this Drive has one (only papers that exist), otherwise all 20,000
   assigned IDs (IDs that do not exist come back as `not_found`, one request each).
3. `crawl_references` in blocks of 100, with progress. Finished papers (`ok` / `not_found`) are
   skipped, so after a disconnect just run all cells again.
4. Writes **`references_export.zip`** (every `references.json` + `logs/references.csv`).

**Hand-over:** put `references_export.zip` in the main project folder (`MyDrive/arxiv-project/`, next to
`data/`) of the person running 05. Its report step imports it. Exporting again later is fine.

Results go to `MyDrive/arxiv-project/references_run/`, a separate folder, so this also works on the
same Drive as the download without touching its files. Put `S2_API_KEY` in Colab Secrets or in
`MyDrive/arxiv-project/.env`.

## 0. Setup

On Colab this clones branch `VA` and installs the packages. `REFERENCE_LIMIT = 100` gives a quick test.

In [ ]:
# On Colab: clone or update branch VA, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB and not Path("00_config.ipynb").is_file():
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "VA"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                        "https://github.com/thinhelzan842/intro-ds-arxiv-project.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")

RUN_TEST = False         # True also runs 03's own live test
REFERENCE_LIMIT = None   # e.g. 100: only the first IDs, as a quick test

In [ ]:
get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
MAIN_MANIFEST = PROJECT_ROOT / "state" / "discovery_manifest.json"   # from 05, if this Drive has it

# A separate folder, so a references run never writes into the download's folders.
REFERENCES_ROOT = PROJECT_ROOT / "references_run"
DATA_DIR, LOG_DIR, STATE_DIR = REFERENCES_ROOT / "data" / "raw", REFERENCES_ROOT / "logs", REFERENCES_ROOT / "state"
PAPER_LOG = LOG_DIR / "papers.csv"
for folder in (DATA_DIR, LOG_DIR, STATE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

for notebook in ("01_discovery", "03_references"):
    get_ipython().run_line_magic("run", f"-i ./{notebook}.ipynb")

saved = read_json(MAIN_MANIFEST)
if saved:
    reference_ids = [p["id"] for p in saved["papers"] if p.get("versions", 0) > 0]
    print(f"{len(reference_ids):,} papers from the discovery manifest")
else:
    reference_ids = all_assigned_ids()
    print(f"No discovery manifest on this Drive: all {len(reference_ids):,} assigned IDs")
if REFERENCE_LIMIT:
    reference_ids = reference_ids[:REFERENCE_LIMIT]
    print(f"REFERENCE_LIMIT: only the first {len(reference_ids):,}")
print(f"S2 key: {'set' if S2_API_KEY else 'NOT SET (shared pool, slow)'} | one request every {S2_INTERVAL:.0f} s")

## 1. Fetch the references

One batch request per 100 IDs, with progress. Stopping the cell is safe: finished papers are saved and skipped next time.

In [ ]:
started = time.perf_counter()
try:
    for offset in range(0, len(reference_ids), 100):
        crawl_references(reference_ids[offset:offset + 100], verbose=False)
        done = min(offset + 100, len(reference_ids))
        minutes = (time.perf_counter() - started) / 60
        print(f"[references] {done:,}/{len(reference_ids):,} IDs checked | {minutes:.1f} min | "
              f"requests {REFERENCE_STATS['requests']:,}, HTTP 429 {REFERENCE_STATS['rate_limited']:,}", flush=True)
except KeyboardInterrupt:
    print("Interrupted: finished papers are saved; run all cells again to continue.")

print(json.dumps(references_summary(reference_ids), indent=2, ensure_ascii=False))

## 2. Export for the 05 runner

Run this cell whenever you want to hand over what is finished so far; the 05 report imports it.

In [ ]:
import zipfile

REFERENCES_EXPORT = PROJECT_ROOT / "references_export.zip"


def export_references(zip_path=REFERENCES_EXPORT):
    """Every references.json under DATA_DIR + logs/references.csv, in one zip for the 05 runner."""
    zip_path = Path(zip_path)
    temporary = zip_path.with_name(zip_path.name + ".tmp")
    files = sorted(DATA_DIR.glob("*/references.json"))
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as package:
        for path in files:
            package.write(path, f"data/raw/{path.parent.name}/references.json")
        if REF_LOG.is_file():
            package.write(REF_LOG, "logs/references.csv")
    os.replace(temporary, zip_path)
    return len(files)


count = export_references()
print(f"{REFERENCES_EXPORT}: {count:,} papers, {REFERENCES_EXPORT.stat().st_size / 1e6:.1f} MB")
print("Hand-over: copy this file to the main project folder of the person running 05_run_pipeline.ipynb")